# 01 — The four rulers

What each ruler measures, and what truth itself scores on it. The truth row is the
calibration: every model column is read against it.

Two of them do **not** read 1.00 at truth and must not be read as though they did.
`climate`'s denominator is two disjoint halves of one finite pool, which pushes them apart,
so an independent draw scores below 1 — `climate_vs_truth` divides that out. And on the SDE
the floor is realisation-vs-realisation noise, which is irreducible: no model and no solver
can get under it.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
for key in DATA:
    r = gt['datasets'][key]
    print(f"--- {key.upper()}  ({r['kind']}, b={r['b']:g}) ---")
    print(f"  horizon   ground truth usable {r['floor_steps']} steps; same-cost solver {r['euler_bar_steps']}"
          + ("   (floor = realisation noise, not discretisation)" if r['kind'] == 'sde' else ""))
    if r['spread_lead'] > 0:
        print(f"  spread    read at lead {r['spread_lead']} steps; a second independent truth "
              f"ensemble scores {r['truth_spread']:.3f} against the first")
    else:
        print("  spread    not defined (truth's ensemble spread is identically zero)")
    print(f"  climate   truth scores {r['truth_climate']:.2f} over 5 independent draws, "
          f"range {r['truth_climate_lo']:.2f}-{r['truth_climate_hi']:.2f}  <- the resolution")
    print(f"  chaos     lambda_1 of the true map = {gt['lambda_true']:.4f}, so the ratio is 1.00 by definition")
    print(f"  alive     truth scores {r['truth_alive']:.2f}   ·   lobe {r['truth_lobe']:.3f} switches/tau")

c = json.load(open(ARTIFACTS / 'summary.json'))['controls']
print("\n--- the k=1 control: what this suite CANNOT resolve ---")
print("same model, same loss, differing only by float summation order (~7e-7 in the weights)")
for name, cc in sorted(c.items()):
    bits = '  '.join(f"{k} {v['rel']*100:5.1f}%" for k, v in cc.items())
    print(f"  {name:12s} {bits}")

--- ODE  (ode, b=0) ---
  horizon   ground truth usable 362 steps; same-cost solver 32
  spread    not defined (truth's ensemble spread is identically zero)
  climate   truth scores 0.62 over 5 independent draws, range 0.56-1.89  <- the resolution
  chaos     lambda_1 of the true map = 0.8974, so the ratio is 1.00 by definition
  alive     truth scores 1.00   ·   lobe 0.631 switches/tau
--- SDE  (sde, b=0.6) ---
  horizon   ground truth usable 23 steps; same-cost solver 26   (floor = realisation noise, not discretisation)
  spread    read at lead 12 steps; a second independent truth ensemble scores 1.022 against the first
  climate   truth scores 0.66 over 5 independent draws, range 0.57-1.00  <- the resolution
  chaos     lambda_1 of the true map = 0.8974, so the ratio is 1.00 by definition
  alive     truth scores 1.00   ·   lobe 0.836 switches/tau
--- SDE015  (sde, b=0.15) ---
  horizon   ground truth usable 104 steps; same-cost solver 29   (floor = realisation noise, not discretisa

## Findings

_Written after reading the numbers above._

- 
- 
- 